# Week 1R — ProsQA completion run — latent-agent-cost

Completion run for the Week-1R ProsQA explicit-CoT baseline (Qwen2.5-1.5B-Instruct, ProsQA, label `WEEK1R`, same corrected protocol as the original run).

**Why this notebook exists:** the combined Week-1R run (Version 3 of `week1-baselines.ipynb`, GSM8K + ProsQA in one commit) was killed at Kaggle's ~12 h session limit during the ProsQA cell. Because the runner checkpoints every attempt to its `.attempts.jsonl`, the ProsQA attempts log survived with **1,449 of 1,500 attempts** — all of tasks 0..482 x seeds {0, 1, 2}. This notebook runs **only the missing 51 attempts** (tasks 483..499 x seeds {0, 1, 2}) via `scripts/run_completion.py`, which loads the same task list and slices it by position, so prompts, verifier, and protocol are identical. Expected duration: minutes-to-tens-of-minutes, not hours — 51 attempts at the v3 pace.

**After this run:** download `results/week1r_prosqa_completion.attempts.jsonl` from the Output tab. The full Week-1R ProsQA table is then produced locally by merging it with the recovered log (`results/week1r_prosqa.attempts.partial.jsonl`) using `scripts/aggregate_attempts.py` — the aggregator validated to reproduce the Week-1R GSM8K table byte for byte from its attempts log.

**Before running:** notebook Settings -> **Accelerator = GPU** (T4 or P100) and **Internet = ON** (needed for the git clone, the ProsQA download, and the model download).

Run all cells top to bottom — or better, use **Save Version -> Save & Run All (Commit)** so the run executes headless and the outputs persist (see `RUNBOOK.md` in this folder).

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU visible. Set Accelerator = GPU in the notebook settings and re-run.")
print("device:", torch.cuda.get_device_name(0))

Clone the project (fresh every run) and install its dependencies. `requirements.txt` already includes `tokenecon==0.2.0`; the one local pin filtered out is `torch` — Kaggle's preinstalled CUDA build must be kept, never replaced by the local CPU pin.

In [ ]:
!rm -rf latent-agent-cost
!git clone https://github.com/karmendra8386/latent-agent-cost
%cd latent-agent-cost
!grep -v '^torch' requirements.txt > /tmp/req-kaggle.txt
!pip install -q -r /tmp/req-kaggle.txt
!python -c "import torch; print('after install - torch:', torch.__version__, '| cuda:', torch.cuda.is_available())"

ProsQA completion slice — tasks 483..499 x seeds {0, 1, 2}, label `WEEK1R`. Attempts are appended and flushed to `results/week1r_prosqa_completion.attempts.jsonl` after every attempt, in the identical schema to the recovered log.

In [ ]:
!python scripts/run_completion.py --dataset prosqa --task-start 483 --task-stop 500 --seeds 0 1 2 --model Qwen/Qwen2.5-1.5B-Instruct --device cuda --dtype float16 --out results/week1r_prosqa_completion --label WEEK1R

List the outputs. The `.md` table written here covers only the 51-attempt completion slice (a sanity check) — the Week-1R ProsQA baseline table comes from the local merge described in the header. The file to download is `results/week1r_prosqa_completion.attempts.jsonl`.

In [ ]:
!ls -la results/
!wc -l results/week1r_prosqa_completion.attempts.jsonl
!cat results/week1r_prosqa_completion.md